# R16c step 2 · A stronger R16 grader, picked on held-out training images, measured against the ceiling

Task **R16c** from `Essential_task_delegation/Progress_Tracker.md` (go given on 19 Oct 2026: test R16 more rigorously). Built as a copy of `r16-find-then-grade.ipynb`, which stays unchanged. The finding, the R11 reference and the E-protocol are R16's; only the grader changes.

**Why** (R16c step 1, on your PC)
- With R16's own outlines (SAM 3 on the before-image plus R11's missed pieces), a perfect grader would reach **Destroyed IoU 0.614 and mIoU 0.766**. R16b's corrected grader gets 0.208 and 0.541. The outlines are good enough; the grader loses most of the score.
- Counting says the same: on the 1,055 validation counting questions, R16's best map answers 36.9% against 39.2% for R11's blobs, and 55.3% with a perfect grader.

**What changes from R16**

| | R16 (and R16b) | R16c |
|---|---|---|
| Crop and context | 96 px, 16 px margin | 128 px, 32 px margin |
| Training | 10 epochs, the last kept | 20 epochs, the best kept |
| Grade decision | the largest probability (R16b: times the real shares) | a bias per grade |
| Who picks epoch and biases | nobody (R16b: a fixed rule) | 580 held-out training images (10%) |
| Seeds | one | two (42, 43) |
| Training buildings | the true pieces | step 2: the true pieces; step 5 (its own notebook): SAM 3's outlines |

- **Held-out training images:** 10% of the 5,798 training images never train the grader.
  - After each epoch the grader grades all their buildings, uncapped, so the mix is the real one.
  - The biases that maximise the mean IoU of the three grades there (pixel-weighted) are found, and the best epoch is kept.
  - The 645 validation images are used once, at the end, as in every D4 run.
- **Step 5 (`TRAIN_OUTLINES = "sam3"`)** has its own notebook, `r16c-step5-grader-on-sam3-outlines.ipynb`, which needs step 4 (`r16c-step4-sam3-training-images.ipynb`) first.
  - Each SAM 3 outline is then labelled by its main true class, so the grader learns on the outlines it is judged on.
  - Outlines that are mostly background get a 4th label, "None", painted as background (the "perfect grader + rejects" ceiling).

**What it reports** (E-protocol: 645 validation images, full resolution, dataset-level confusion matrix)
- Per seed, R16's three maps:
  - SAM 3 only;
  - SAM 3 + R11's pieces;
  - the vote with R11.

  Each comes with the calibrated biases, and with R16b's fixed real-share rule for reference.
- The ceiling on the same outlines (a perfect grader, with and without rejects), and R11 with and without TTA. R11 is re-run and must reproduce `D4_R11.csv`.
- Each grader on the true validation buildings (object level), against R16's macro F1 of 0.656.
- **Keep rule** against R11 with TTA, on both seeds: Destroyed IoU up by more than 0.013, mIoU down by at most 0.005. Also the Phase Gate, and how much of the gap from R16b to the ceiling each map closes.

**How to run on Kaggle** (account 1, the one that ran E3)
1. Settings: GPU T4 ×1, internet ON, secret `HF_TOKEN` (R11's checkpoint is in the private repo `AbrarAlam/disasterm3-d4-R11`).
2. Inputs:
   - the DisasterM3 mirror (attached automatically);
   - E3's output: Add Input → Your Work → `e3-sam3-building-finding`;
3. Run All once with `SMOKE_TEST = True` (20 training images, 8 validation images, 2 epochs, one seed). Then set it to `False` and use Save & Run All, about 2 hours.
4. Outputs in `/kaggle/working/R16c-truth/`:
   - `D4_R16c.csv`, `R16c_summary.csv` (also per source), `grader_true_buildings.csv`;
   - `instances_val.csv`: each found building with both seeds' log-probabilities and its true and R11 pixels;
   - `calibration.json` (the kept epoch and biases), `history.json`, `grader_seed42.pth` and `grader_seed43.pth`;
   - `config.json`, `README.md`.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed only to load R11 (its checkpoint key layout).
# The grader uses torchvision, which Kaggle already has.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.6 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [ ]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import glob
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
GRADES = CLASS_NAMES[1:]
SMOKE_TEST = False                  # True = 20 training images (4 held out), 8 validation images, 2 epochs, one seed

# Step 2 trains on the true building pieces. Step 5 (r16c-step5-grader-on-sam3-outlines.ipynb) is this notebook with "sam3", after
# r16c-step4-sam3-training-images.ipynb (step 4) has run SAM 3 on the training images: the grader then learns on the
# outlines it is judged on, plus a 4th label, "None", for SAM 3 outlines that are mostly background.
TRAIN_OUTLINES = "truth"            # "truth" (step 2) or "sam3" (step 5)
LABELS = GRADES + (["None"] if TRAIN_OUTLINES == "sam3" else [])
OUT_DIR = Path(f"/kaggle/working/R16c-{TRAIN_OUTLINES}")
CACHE_DIR = Path("/kaggle/temp/r16c_crops")   # crops (about 13 GB for step 2), not saved as output

CROP = 128                          # R16: 96. Larger buildings keep more detail
MARGIN = 32                         # R16: 16. More context around small buildings
MIN_AREA = 20                       # building pieces smaller than this are skipped (R16's value)
MAX_PER_IMAGE = {"Intact": 12, "None": 4}   # training images only; damaged and destroyed buildings are all kept
EPOCHS, BATCH, LR, WEIGHT_DECAY = 20, 256, 3e-4, 1e-4
SEEDS = (42, 43)
HOLD_OUT = 0.1                      # training images held out to pick the epoch and the grade biases
BIAS_GRID = np.round(np.arange(-4, 4.001, 0.25), 2)   # log-odds added to one label's score (intact stays at 0)
NUM_WORKERS = 4

# The U-Net for the footprint merge, the vote and the reference rows, as in R16. R11 never trained on
# benchmark pictures, so these outputs may answer benchmark questions.
UNET = "R11"
EXPECTED_UNET = {"std": (0.5811, 0.2911), "tta": (0.5875, 0.2965)}   # D4_R11.csv: (mIoU, Destroyed IoU)
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth
KEEP_RULE = {"min_destroyed_gain": 0.013, "max_miou_drop": 0.005}     # against the U-Net with TTA
R16B = {"sam3": (0.5150, 0.2279), "sam3+r11": (0.5407, 0.2079), "vote": (0.5846, 0.2737)}   # R16b: (mIoU, Destroyed IoU)

if SMOKE_TEST:
    EPOCHS, BATCH, SEEDS = 2, 32, SEEDS[:1]

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add a Kaggle dataset with {name}/final_model.pth.")

def find_dir(name, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No '{name}' folder under /kaggle/input: {why}")
    return Path(hits[0])

# The finding on the validation images: E3's SAM 3 maps of the before-image, 2x2 grid (16-bit PNG, 0 = background)
SAM3_DIR = find_dir("pre_grid2", "add E3's output (e3-sam3-building-finding), or a dataset with its masks/pre_grid2/")
SAM3_TRAIN_DIR = (find_dir("pre_grid2_train", "add the output of r16c-step4-sam3-training-images")
                  if TRAIN_OUTLINES == "sam3" else None)
UNET_CKPT = find_checkpoint(UNET)

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ R16c, grader trained on {'the true pieces (step 2)' if TRAIN_OUTLINES == 'truth' else 'SAM 3 outlines (step 5)'}"
      f"{'  [SMOKE TEST]' if SMOKE_TEST else ''}: labels {LABELS}; {CROP} px crops, {MARGIN} px margin; "
      f"{EPOCHS} epochs; seeds {SEEDS}")
print(f"  Finding (SAM 3): {SAM3_DIR} ({len(list(SAM3_DIR.glob('*.png')))} maps)")
if SAM3_TRAIN_DIR:
    print(f"  SAM 3 on the training images: {SAM3_TRAIN_DIR} ({len(list(SAM3_TRAIN_DIR.glob('*.png'))):,} maps)")
print(f"  U-Net {UNET}: {UNET_CKPT}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ R16c, grader trained on the true pieces (step 2): labels ['Intact', 'Damaged', 'Destroyed']; 128 px crops, 32 px margin; 20 epochs; seeds (42, 43)
  Stage 1 (SAM 3): /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2 (645 maps)
  U-Net R11: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth
  Device: Tesla T4; outputs: /kaggle/working/R16c-truth


## 1. Data: the same 6,443 pairs and the same 645 validation images as every D4 run
Cell 3 is copied verbatim from R11 (via R16), so the split matches R1–R16 exactly and every pair keeps its before-image. Cell 4 then holds out 10% of the training images, which never train the grader.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from R11's Cell 4) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Before: {pairs[0]['pre_image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Before: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_pre_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [ ]:
# ── Cell 4: Validation split (identical to every D4 run), the held-out training images, and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
fit_pairs, hold_pairs = train_test_split(train_pairs, test_size=HOLD_OUT, random_state=7)   # never validation images

def sam_train_path(pair):
    return SAM3_TRAIN_DIR / f"{Path(pair['image_path']).stem}.png"

if TRAIN_OUTLINES == "sam3":        # only training images whose SAM 3 map exists
    no_map = sum(not sam_train_path(p).exists() for p in train_pairs)
    fit_pairs = [p for p in fit_pairs if sam_train_path(p).exists()]
    hold_pairs = [p for p in hold_pairs if sam_train_path(p).exists()]
    assert SMOKE_TEST or no_map <= 0.02 * len(train_pairs), f"{no_map:,} training images have no SAM 3 map: finish step 4"
if SMOKE_TEST:   # 20 grader images, those with destroyed pixels first, so every grade gets crops
    with_destroyed = [p for p in fit_pairs[:400] if (cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE) == 3).any()]
    fit_pairs = (with_destroyed + [p for p in fit_pairs if p not in with_destroyed])[:20]
    hold_pairs, val_pairs = hold_pairs[:4], val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in train_pairs + val_pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Grader training images: {len(fit_pairs):,}; held out: {len(hold_pairs):,}; validation: {len(val_pairs)} "
      f"({n_xbd} xBD, {len(val_pairs) - n_xbd} other). Expected 5,218, 580 and 645 (560 xBD, 85 other) "
      "for the full step-2 run.")

✓ Grader training images: 5,218; held out: 580; validation: 645 (560 xBD, 85 other). Expected 5,218, 580 and 645 (560 xBD, 85 other) for the full step-2 run.


## 2. One crop per building
Every building becomes the same thing as in R16: a square crop around its box, resized to 128 px. Each crop holds the before-image (3 channels), the after-image (3) and the building's own mask (1). The margin is now 32 px.
- **Step 2 (`truth`):** a training building is an 8-connected piece of one true class, as in R16.
- **Step 5 (`sam3`, its own notebook):** a training building is one SAM 3 outline, labelled by the true class most of its pixels have; mostly-background outlines become "None".
- **Caps:** the grader's training images keep every damaged and destroyed building, at most 12 intact and 4 "None" per image. The held-out images keep everything, so the biases are picked on the real mix.

In [ ]:
# ── Cell 5: Crop helpers, and the buildings of a training image ────────────
from scipy import ndimage

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side

def building_crops(pre, post, labels, ids):
    """labels: int map of building ids (0 = none). Returns uint8 (len(ids), 7, CROP, CROP):
    before RGB, after RGB, and 255 where the pixel belongs to that building."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        top, left, side = square_box(slices[lid - 1], h, w)
        win = (slice(top, top + side), slice(left, left + side))
        out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 3:6] = cv2.resize(post[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP),
                               interpolation=cv2.INTER_NEAREST)
    return out

def read_sam(path, shape):
    """A SAM 3 instance map (16-bit PNG, 0 = nothing) at the image's size; zeros if it is missing."""
    sam = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if sam is None:
        return np.zeros(shape, np.int32)
    if sam.shape != shape:
        sam = cv2.resize(sam, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return sam.astype(np.int32)

def label_of(cls):
    """True class 1-3 -> its label (0-2); 0, mostly background -> "None" (step 5 only)."""
    return cls - 1 if cls else LABELS.index("None")

def pieces(gt, sam=None):
    """The buildings of one image as (label, id map, ids, true pixels per building (n, 4)).
    Without sam: the 8-connected pieces of each true class of at least MIN_AREA px (R16's rule).
    With sam: SAM 3's outlines of at least MIN_AREA px, each labelled by the true class most of its pixels have."""
    out = []
    if sam is None:
        for cls in (1, 2, 3):
            n, labels, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
            ids = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]
            if ids:
                px = np.zeros((len(ids), 4), np.int64)
                px[:, cls] = stats[ids, cv2.CC_STAT_AREA]
                out.append((label_of(cls), labels, ids, px))
        return out
    size = int(sam.max()) + 1
    px_all = np.bincount(sam.ravel().astype(np.int64) * 4 + gt.ravel(), minlength=size * 4).reshape(size, 4)
    main = px_all.argmax(1)
    big = px_all.sum(1) >= MIN_AREA
    for cls in (1, 2, 3, 0):
        ids = [int(i) for i in np.flatnonzero(big & (main == cls)) if i > 0]
        if ids:
            out.append((label_of(cls), sam, ids, px_all[ids]))
    return out

# Self-checks: a box at the image corner stays inside, each crop marks only its own building, the pieces rules
_lab = np.zeros((200, 300), np.int32)
_lab[0:10, 0:20] = 1
_lab[100:150, 200:260] = 2
assert square_box(ndimage.find_objects(_lab)[0], 200, 300) == (0, 0, 20 + 2 * MARGIN)
_crops = building_crops(np.zeros((200, 300, 3), np.uint8), np.zeros((200, 300, 3), np.uint8), _lab, [1, 2])
assert _crops.shape == (2, 7, CROP, CROP) and _crops[1, 6].max() == 255 and _crops[0, 6].mean() < 255
_gt = np.zeros((200, 300), np.uint8)
_gt[0:10, 0:20], _gt[100:150, 200:260], _gt[190:192, 0:2] = 3, 1, 2      # the 4-pixel piece is below MIN_AREA
assert [(l, len(i)) for l, _, i, _ in pieces(_gt)] == [(0, 1), (2, 1)]
if TRAIN_OUTLINES == "sam3":
    _sam = np.zeros((200, 300), np.int32)
    _sam[0:10, 0:20], _sam[100:150, 200:260], _sam[160:180, 0:30] = 1, 2, 3   # destroyed, intact, background
    assert [(LABELS[l], i) for l, _, i, _ in pieces(_gt, _sam)] == [("Intact", [2]), ("Destroyed", [1]), ("None", [3])]
print("✓ Crop helpers ready (self-checks passed)")

✓ Crop helpers ready (self-checks passed)


In [6]:
# ── Cell 6: Crops of the training buildings → uint8 files on disk ──────────
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED_DATA = 42                      # the capped sample of intact (and "None") buildings is the same in every run

class ImageCrops(Dataset):
    """All building crops of one image, their labels and true pixels; caps limit some labels per image."""

    def __init__(self, pairs_subset, caps=None):
        self.pairs, self.caps = pairs_subset, caps or {}

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        crops, labels, px = [np.zeros((0, 7, CROP, CROP), np.uint8)], [], [np.zeros((0, 4), np.int64)]
        pair = self.pairs[idx]
        data = read_triplet(pair)
        if data is not None:
            post, pre, gt = data
            sam = read_sam(sam_train_path(pair), gt.shape) if TRAIN_OUTLINES == "sam3" else None
            rng = np.random.default_rng(SEED_DATA + idx)    # the same sample whichever worker reads the image
            for label, id_map, ids, pix in pieces(gt, sam):
                cap = self.caps.get(LABELS[label])
                if cap and len(ids) > cap:
                    keep = np.sort(rng.choice(len(ids), cap, replace=False))
                    ids, pix = [ids[k] for k in keep], pix[keep]
                crops.append(building_crops(pre, post, id_map, ids))
                labels += [label] * len(ids)
                px.append(pix)
        return np.concatenate(crops), np.array(labels, np.int64), np.concatenate(px)

def build_cache(pairs_subset, name, caps=None):
    """Writes every crop to CACHE_DIR/<name>.u8; returns that file, the labels and the true pixels per building."""
    path, labels, px = CACHE_DIR / f"{name}.u8", [], []
    loader = DataLoader(ImageCrops(pairs_subset, caps), batch_size=None, num_workers=NUM_WORKERS)
    with open(path, "wb") as f:
        for crops, lab, pix in tqdm(loader, desc=f"Crops ({name})"):
            f.write(np.asarray(crops, np.uint8).tobytes())
            labels.append(np.asarray(lab, np.int64))
            px.append(np.asarray(pix, np.int64))
    labels, px = np.concatenate(labels), np.concatenate(px)
    assert path.stat().st_size == len(labels) * 7 * CROP * CROP, "cache size and label count disagree"
    return path, labels, px

t0 = time.time()
FIT_CACHE, fit_labels, _ = build_cache(fit_pairs, "fit", MAX_PER_IMAGE)
HOLD_CACHE, hold_labels, hold_px = build_cache(hold_pairs, "hold")      # every building: the real mix
fit_counts = np.bincount(fit_labels, minlength=len(LABELS))
hold_counts = np.bincount(hold_labels, minlength=len(LABELS))
REAL_SHARE = (hold_counts + 1) / (hold_counts.sum() + len(LABELS))     # for R16b's fixed rule (add-one, never 0)
print(f"✓ {len(fit_labels):,} grader-training buildings and {len(hold_labels):,} held-out ones in "
      f"{(time.time() - t0) / 60:.1f} min ({(FIT_CACHE.stat().st_size + HOLD_CACHE.stat().st_size) / 1e9:.1f} GB)")
print("  training (capped): " + ", ".join(f"{l} {c:,}" for l, c in zip(LABELS, fit_counts)))
print("  held out (all):    " + ", ".join(f"{l} {c:,} ({s:.1%})" for l, c, s in zip(LABELS, hold_counts, REAL_SHARE)))
assert fit_counts.min() > 0, "a label has no training buildings"

Crops (fit):   0%|          | 0/5218 [00:00<?, ?it/s]

Crops (hold):   0%|          | 0/580 [00:00<?, ?it/s]

✓ 92,554 grader-training buildings and 22,194 held-out ones in 4.9 min (13.2 GB)
  training (capped): Intact 38,479, Damaged 37,892, Destroyed 16,183
  held out (all):    Intact 16,367 (73.7%), Damaged 3,889 (17.5%), Destroyed 1,938 (8.7%)


## 3. The grading: the Siamese grader, one per seed
The same network as R16: one ResNet-18 (ImageNet weights) reads the before crop and the after crop, each with the building's mask as a 4th channel, and the head sees `[before, after, after − before]`. Training draws each label equally often and flips and rotates the crops.

After every epoch:
1. The grader grades all the held-out buildings.
2. A coordinate search over a bias per label (intact fixed at 0, −4 to +4 in steps of 0.25) maximises the mean IoU of the three grades. Each building adds its true pixels to the column of its grade.
3. The epoch with the best score is kept with its biases.

R16b's fixed rule is scored alongside for reference: each label times its real share, divided by its training share.

In [ ]:
# ── Cell 7: The grading, the Siamese grader per seed; the epoch and the label biases picked on held-out images ──
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import WeightedRandomSampler

class CropSet(Dataset):
    """Crops from a cache file. The memmap opens inside each worker, so it is never pickled."""

    def __init__(self, path, labels, augment):
        self.path, self.labels, self.augment, self.data = path, labels, augment, None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        if self.data is None:
            self.data = np.memmap(self.path, np.uint8, "r", shape=(len(self.labels), 7, CROP, CROP))
        x = np.array(self.data[i])
        if self.augment:   # the same flip and rotation for the before crop, the after crop and the mask
            x = np.rot90(x, random.randrange(4), axes=(1, 2))
            if random.random() < 0.5:
                x = x[:, :, ::-1]
        return torch.from_numpy(np.ascontiguousarray(x)), int(self.labels[i])

class SiameseGrader(nn.Module):
    """A shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> one score per label."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        rgb = net.conv1.weight.data
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.conv1.weight.data = torch.cat([rgb, rgb.mean(1, keepdim=True)], 1)   # the mask channel starts as grey
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2),
                                  nn.Linear(256, len(LABELS)))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)

@torch.no_grad()
def log_probs(model, crops):
    """uint8 crops (N, 7, S, S), an array or a memmap -> log-probabilities (N, labels)."""
    model.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.log_softmax(model(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, len(LABELS)), np.float32)

def to_class(labels):
    """Grader labels -> mask classes: intact, damaged, destroyed -> 1-3; "None" -> 0 (background)."""
    return np.where(labels < 3, labels + 1, 0)

def grade_ious(labels, px):
    """IoU of each mask class over a set of buildings, each adding its true pixels to the column of its label."""
    cls = to_class(labels)
    cm = np.zeros((4, 4))
    for k in range(4):
        cm[:, k] = px[cls == k].sum(0)
    tp = np.diag(cm)
    return tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)

def best_bias(logp, px, rounds=3):
    """The label biases (intact fixed at 0) that maximise the mean IoU of the three grades, by coordinate search."""
    bias = np.zeros(len(LABELS))
    score = grade_ious((logp + bias).argmax(1), px)[1:].mean()
    for _ in range(rounds):
        for j in range(1, len(LABELS)):
            for b in BIAS_GRID:
                trial = bias.copy()
                trial[j] = b
                s = grade_ious((logp + trial).argmax(1), px)[1:].mean()
                if s > score + 1e-9:
                    score, bias = s, trial
    return float(score), bias

# Self-check: the search finds a bias that moves the second building from damaged to destroyed
_lp = np.full((2, len(LABELS)), -9.0)
_lp[0, 0], _lp[1, 1], _lp[1, 2] = -0.1, -0.5, -1.0
_px = np.zeros((2, 4), np.int64)
_px[0, 1], _px[1, 3] = 100, 50                     # truly intact, truly destroyed
assert (_lp + best_bias(_lp, _px)[1]).argmax(1).tolist() == [0, 2]

fit_set = CropSet(FIT_CACHE, fit_labels, augment=True)
hold_mm = np.memmap(HOLD_CACHE, np.uint8, "r", shape=(len(hold_labels), 7, CROP, CROP))
weights = torch.tensor(1.0 / fit_counts[fit_labels], dtype=torch.double)     # each label drawn equally often
PRIOR_BIAS = np.log(REAL_SHARE) - np.log(1 / len(LABELS))                    # R16b's fixed rule, for reference
PRIOR_BIAS -= PRIOR_BIAS[0]                                                   # intact at 0, like the searched biases

graders, BIASES, CHOSEN, histories = {}, {}, {}, {}
t_train = time.time()
for seed in SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    sampler = WeightedRandomSampler(weights, num_samples=len(fit_set), replacement=True,
                                    generator=torch.Generator().manual_seed(seed))
    loader = DataLoader(fit_set, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True, persistent_workers=True)
    grader = SiameseGrader().to(DEVICE)
    optimizer = torch.optim.AdamW(grader.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best, history, t0 = {"score": -1.0}, [], time.time()
    for epoch in range(EPOCHS):
        grader.train()
        total, correct, seen = 0.0, 0, 0
        for x, y in tqdm(loader, desc=f"Seed {seed}, epoch {epoch + 1}/{EPOCHS}", leave=False):
            before, after = to_inputs(x)
            y = y.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = grader(before, after)
            loss = F.cross_entropy(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item() * len(y)
            correct += int((logits.argmax(1) == y).sum())
            seen += len(y)
        logp = log_probs(grader, hold_mm)
        score, bias = best_bias(logp, hold_px)
        shares_score = float(grade_ious((logp + PRIOR_BIAS).argmax(1), hold_px)[1:].mean())
        history.append({"epoch": epoch + 1, "loss": total / max(seen, 1),
                        "balanced_train_accuracy": correct / max(seen, 1), "held_out_grade_iou": round(score, 4),
                        "held_out_grade_iou_real_shares": round(shares_score, 4), "bias": bias.round(2).tolist()})
        print(f"Seed {seed} | ep {epoch + 1:>2}/{EPOCHS} | loss {history[-1]['loss']:.4f} | held-out mean grade IoU "
              f"{score:.4f} (real shares {shares_score:.4f}) | {(time.time() - t0) / 60:.1f} min")
        if score > best["score"]:
            best = {"score": score, "epoch": epoch + 1, "bias": bias,
                    "state": {k: v.detach().cpu().clone() for k, v in grader.state_dict().items()}}
    del loader
    grader.load_state_dict(best["state"])
    grader.eval()
    torch.save(best["state"], OUT_DIR / f"grader_seed{seed}.pth")
    graders[seed], BIASES[seed], histories[seed] = grader, best["bias"], history
    CHOSEN[seed] = {"epoch": best["epoch"], "held_out_grade_iou": round(best["score"], 4),
                    "bias": dict(zip(LABELS, best["bias"].round(2).tolist()))}
    print(f"✓ Seed {seed}: kept epoch {best['epoch']} (held-out mean grade IoU {best['score']:.4f}), "
          f"biases {CHOSEN[seed]['bias']}")
GRADER_MIN = (time.time() - t_train) / 60
print(f"✓ {len(SEEDS)} grader(s) trained in {GRADER_MIN:.1f} min. R16b's fixed rule for reference: "
      f"{dict(zip(LABELS, PRIOR_BIAS.round(2).tolist()))}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


Seed 42, epoch 1/20:   0%|          | 0/361 [00:00<?, ?it/s]

/tmp/ipykernel_23/297587102.py:60: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))


Seed 42 | ep  1/20 | loss 0.4823 | held-out mean grade IoU 0.6410 (real shares 0.6243) | 1.5 min


...

Seed 43 | ep 18/20 | loss 0.1039 | held-out mean grade IoU 0.6105 (real shares 0.5674) | 26.7 min


Seed 43, epoch 19/20:   0%|          | 0/361 [00:00<?, ?it/s]

Seed 43 | ep 19/20 | loss 0.1011 | held-out mean grade IoU 0.6123 (real shares 0.5640) | 28.2 min


Seed 43, epoch 20/20:   0%|          | 0/361 [00:00<?, ?it/s]

Seed 43 | ep 20/20 | loss 0.0975 | held-out mean grade IoU 0.6116 (real shares 0.5669) | 29.7 min
✓ Seed 43: kept epoch 6 (held-out mean grade IoU 0.6638), biases {'Intact': 0.0, 'Damaged': -2.0, 'Destroyed': -4.0}
✓ 2 grader(s) trained in 59.4 min. R16b's fixed rule for reference: {'Intact': 0.0, 'Damaged': -1.44, 'Destroyed': -2.13}


## 4. Validation, once: the U-Net, the graders on the true buildings, the R16 maps and the ceiling
For each of the 645 validation images:
1. **R11** predicts with and without TTA (the D4 code). Its scores must reproduce `D4_R11.csv`.
2. **Each grader on the true buildings**, with its own biases, for the object-level scores.
3. **The finding, as in R16:** SAM 3's instances, plus R11's building pieces that SAM 3 missed (at least 20 px).
4. **The ceiling:** a perfect grader on the same outlines (each building's main true grade; "rejects" may also say background).
5. **Each seed's maps:** SAM 3 only, SAM 3 + R11's pieces, and the vote with R11. Each is scored with the searched biases and with R16b's fixed rule.

In [8]:
# ── Cell 8: The U-Net (the D4 prediction code) and the E-protocol metrics ───
import segmentation_models_pytorch as smp

def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall + eps),
            "f2": 5 * precision * recall / (4 * precision + recall + eps),
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
unet = load_unet(UNET_CKPT)
print(f"✓ {UNET} loaded from {UNET_CKPT} (metric self-check passed)")

✓ R11 loaded from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth (metric self-check passed)


In [ ]:
# ── Cell 9: Validation, image by image ─────────────────────────────────────
U = UNET.lower()
MAPS = ["sam3", f"sam3+{U}", "vote"]
RULES = {"": "searched biases", "_shares": "R16b's fixed rule"}
VARIANTS = ([f"{UNET}_std", f"{UNET}_tta", "ceiling_sam3", f"ceiling_sam3+{U}", f"ceiling_rejects_sam3+{U}"]
            + [f"R16c_s{seed}_{m}{rule}" for seed in SEEDS for rule in RULES for m in MAPS])
SOURCES = ("all", "xBD", "other")
cms = {v: {s: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64) for s in SOURCES} for v in VARIANTS}
true_cm = {seed: np.zeros((3, len(LABELS)), np.int64) for seed in SEEDS}   # rows: true grade, columns: label
inst_rows, missing_sam = [], []

t0 = time.time()
for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc="Validation"):
    data = read_triplet(pair)
    if data is None:
        continue
    post, pre, gt = data
    h, w = gt.shape
    stem = Path(pair["image_path"]).stem

    # 1) The U-Net, standard and 8-view TTA (R16 already saved its TTA masks)
    pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, pre))

    # 2) Each grader on the true buildings (object level), with its own biases
    true_parts = pieces(gt)
    if true_parts:
        true_crops = np.concatenate([building_crops(pre, post, lab, ids) for _, lab, ids, _ in true_parts])
        true_labels = np.concatenate([np.full(len(ids), l) for l, _, ids, _ in true_parts])
        for seed, g in graders.items():
            np.add.at(true_cm[seed], (true_labels, (log_probs(g, true_crops) + BIASES[seed]).argmax(1)), 1)

    # 3) The finding, as in R16: SAM 3's instances, then the U-Net's building pieces that SAM 3 missed
    sam = cv2.imread(str(SAM3_DIR / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
    if sam is None:
        missing_sam.append(stem)
        sam = np.zeros((h, w), np.uint16)
    if sam.shape != (h, w):
        sam = cv2.resize(sam, (w, h), interpolation=cv2.INTER_NEAREST)
    ids = sam.astype(np.int32)
    n_sam = int(ids.max())
    n, extra, stats, _ = cv2.connectedComponentsWithStats(((pred_tta > 0) & (ids == 0)).astype(np.uint8),
                                                          connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= MIN_AREA
    keep[0] = False
    new_id = np.zeros(n, np.int32)
    new_id[keep] = n_sam + 1 + np.arange(int(keep.sum()))
    ids = np.where(extra > 0, new_id[extra], ids)          # ids above n_sam are the U-Net's pieces

    # 4) The ceiling on these outlines, then each seed's grades and its three maps
    present = np.unique(ids)
    present = present[present > 0]
    size = int(ids.max()) + 1
    is_present = np.zeros(size, bool)
    is_present[present] = True
    from_unet = np.arange(size) > n_sam
    flat = ids.ravel().astype(np.int64) * NUM_CLASSES
    unet_px = np.bincount(flat + pred_tta.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    gt_px = np.bincount(flat + gt.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    unet_bld = unet_px[:, 1:].astype(np.float32)
    unet_sum = unet_bld.sum(1, keepdims=True)
    perfect = np.where(is_present, gt_px[:, 1:].argmax(1) + 1, 0).astype(np.uint8)   # the main true grade
    rejects = np.where(is_present, gt_px.argmax(1), 0).astype(np.uint8)              # may also say background
    preds = {f"{UNET}_std": pred_std, f"{UNET}_tta": pred_tta,
             "ceiling_sam3": np.where(from_unet, 0, perfect)[ids], f"ceiling_sam3+{U}": perfect[ids],
             f"ceiling_rejects_sam3+{U}": rejects[ids]}
    crops = building_crops(pre, post, ids, present) if len(present) else None
    logps = {}
    for seed, g in graders.items():
        logp = np.zeros((size, len(LABELS)), np.float32)
        if len(present):
            logp[present] = log_probs(g, crops)
        logps[seed] = logp
        for rule, bias in (("", BIASES[seed]), ("_shares", PRIOR_BIAS)):
            scores = logp + bias
            label = scores.argmax(1)
            by_grader = np.where(is_present, to_class(label), 0).astype(np.uint8)
            grade_p = np.exp(scores[:, :3] - scores[:, :3].max(1, keepdims=True))   # the three grades, renormalised
            grade_p /= grade_p.sum(1, keepdims=True)
            shares = np.where(unet_sum > 0, unet_bld / np.maximum(unet_sum, 1), grade_p)   # no U-Net pixel: grader only
            by_vote = np.where(is_present & (label < 3), (0.5 * (grade_p + shares)).argmax(1) + 1, 0).astype(np.uint8)
            preds[f"R16c_s{seed}_sam3{rule}"] = np.where(from_unet, 0, by_grader)[ids]
            preds[f"R16c_s{seed}_sam3+{U}{rule}"] = by_grader[ids]
            preds[f"R16c_s{seed}_vote{rule}"] = by_vote[ids]                   # "None" stays background
    for v, p in preds.items():
        cm = confusion(gt, p)
        cms[v]["all"] += cm
        cms[v][source] += cm

    for lid in present:
        inst_rows.append({"image": f"{stem}.png", "event": event, "source": source, "id": int(lid),
                          "stage1": "sam3" if lid <= n_sam else U, "area_px": int(gt_px[lid].sum()),
                          **{f"logp_{l.lower()}_s{seed}": round(float(logps[seed][lid, j]), 4)
                             for seed in SEEDS for j, l in enumerate(LABELS)},
                          **{f"{U}_{c.lower()}_px": int(unet_px[lid, i]) for i, c in enumerate(CLASS_NAMES)},
                          **{f"gt_{c.lower()}_px": int(gt_px[lid, i]) for i, c in enumerate(CLASS_NAMES)}})

if missing_sam:
    print(f"⚠ {len(missing_sam)} validation images had no SAM 3 map (they got the U-Net's pieces only), "
          f"e.g. {missing_sam[:3]}")
assert len(missing_sam) <= 0.05 * len(val_pairs), "Most SAM 3 maps are missing: is SAM3_DIR E3's pre_grid2 folder?"
VAL_MIN = (time.time() - t0) / 60
print(f"✓ {len(val_pairs)} validation images in {VAL_MIN:.1f} min; {len(inst_rows):,} found buildings graded "
      f"by {len(SEEDS)} grader(s)")

Validation:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 645 validation images in 10.6 min; 57,991 stage-1 buildings graded by 2 grader(s)


## 5. Results, the sanity check, the keep rule, the ceiling and the output files

In [ ]:
# ── Cell 10: Results, the sanity check, the keep rule, the ceiling and the output files ──
import csv
import pandas as pd

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

# 1) Pixel scores per variant, overall and per source
rows = []
for v in VARIANTS:
    for s in SOURCES:
        m = metrics(cms[v][s])
        rows.append({"variant": v, "source": s, "miou": round(m["miou"], 4),
                     **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
                     "building_iou": round(m["building_iou"], 4),
                     "destroyed_precision": round(float(m["precision"][3]), 4),
                     "destroyed_recall": round(float(m["recall"][3]), 4)})
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "R16c_summary.csv", index=False)
overall = summary[summary["source"] == "all"].drop(columns="source")
print(overall.to_string(index=False))

# 2) Sanity check: the U-Net must reproduce its D4 CSV
ok = all(abs(metrics(cms[f"{UNET}_{mode}"]["all"])["miou"] - exp[0]) < 5e-4
         and abs(metrics(cms[f"{UNET}_{mode}"]["all"])["iou"][3] - exp[1]) < 5e-4
         for mode, exp in EXPECTED_UNET.items())
sanity = "n/a (smoke test)" if SMOKE_TEST else ("yes" if ok else "NO")
print(f"\n{UNET} reproduces its D4 CSV: {sanity}")
if sanity == "NO":
    print("⚠ Check the checkpoint (wrong run or not the last epoch) before trusting the comparison.")

# 3) Each grader on the true validation buildings (object level, rows = truth)
true_rows, macro_f1 = [], {}
for seed in SEEDS:
    cm = true_cm[seed].astype(np.float64)
    tp = np.diag(cm[:, :3])
    precision = tp / np.maximum(cm[:, :3].sum(0), 1)
    recall = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
    macro_f1[seed] = round(float(f1.mean()), 4)
    true_rows += [{"seed": seed, "grade": g, "buildings": int(cm[i].sum()), "precision": round(float(precision[i]), 4),
                   "recall": round(float(recall[i]), 4), "f1": round(float(f1[i]), 4),
                   **{f"graded_{l.lower()}": int(true_cm[seed][i, j]) for j, l in enumerate(LABELS)}}
                  for i, g in enumerate(GRADES)]
true_df = pd.DataFrame(true_rows)
true_df.to_csv(OUT_DIR / "grader_true_buildings.csv", index=False)
print(f"\nThe graders on the true validation buildings (macro F1 {macro_f1}; R16 0.656, BDAChat 0.763 on its own split):")
print(true_df.to_string(index=False))

# 4) The keep rule and the Phase Gate against the U-Net with TTA (searched biases), and the share of the gap
#    from R16b to the ceiling on the same outlines that each map closes (Destroyed IoU)
base = metrics(cms[f"{UNET}_tta"]["all"])
ceiling = {"sam3": metrics(cms["ceiling_sam3"]["all"]), f"sam3+{U}": metrics(cms[f"ceiling_sam3+{U}"]["all"])}
verdicts, gates, gap_closed = {}, {}, {}
print(f"\nKeep rule against {UNET} TTA (Destroyed IoU up by more than {KEEP_RULE['min_destroyed_gain']}, "
      f"mIoU down by at most {KEEP_RULE['max_miou_drop']}):")
for seed in SEEDS:
    for m in MAPS:
        v = f"R16c_s{seed}_{m}"
        mm = metrics(cms[v]["all"])
        d_miou, d_destroyed = mm["miou"] - base["miou"], float(mm["iou"][3] - base["iou"][3])
        verdicts[v] = bool(d_destroyed > KEEP_RULE["min_destroyed_gain"] and d_miou >= -KEEP_RULE["max_miou_drop"])
        gates[v] = bool(mm["miou"] > 0.52 and mm["iou"][3] > 0.30)
        line = (f"  {v:>18}: mIoU {mm['miou']:.4f} ({d_miou:+.4f}) | Destroyed IoU {mm['iou'][3]:.4f} "
                f"({d_destroyed:+.4f}) → {'KEEP' if verdicts[v] else 'not kept'} | Phase Gate {'pass' if gates[v] else 'fail'}")
        if m in ceiling:
            r16b = R16B[m.replace(f"+{U}", "+r11")][1]
            gap_closed[v] = round((float(mm["iou"][3]) - r16b) / (float(ceiling[m]["iou"][3]) - r16b), 3)
            line += f" | {gap_closed[v]:.0%} of the gap from R16b ({r16b:.3f}) to the ceiling ({ceiling[m]['iou'][3]:.3f})"
        print(line)
kept_on_all_seeds = {m: all(verdicts[f"R16c_s{seed}_{m}"] for seed in SEEDS) for m in MAPS}
gate_on_all_seeds = {m: all(gates[f"R16c_s{seed}_{m}"] for seed in SEEDS) for m in MAPS}
print(f"Kept on every seed: {kept_on_all_seeds}; Phase Gate on every seed: {gate_on_all_seeds}")

# 5) Output files
pd.DataFrame(inst_rows).to_csv(OUT_DIR / "instances_val.csv", index=False)
(OUT_DIR / "history.json").write_text(json.dumps({str(s): h for s, h in histories.items()}, indent=1))
(OUT_DIR / "calibration.json").write_text(json.dumps({str(s): c for s, c in CHOSEN.items()}, indent=1))

CSV_FIELDS = ["member", "experiment_id", "category", "variable", "value", "miou", "bg_iou",
              "intact_iou", "damaged_iou", "destroyed_iou", "precision", "recall", "f1", "f2",
              "destroyed_precision", "destroyed_recall", "destroyed_f2", "train_time_hrs", "seed", "notes"]

def result_row(exp_id, m, value, notes, seed=None, train_hrs=None):
    return {
        "member": "Phase C (R16c)", "experiment_id": exp_id, "category": "E-protocol",
        "variable": "find-then-grade", "value": value, "miou": round(m["miou"], 4),
        "bg_iou": round(float(m["iou"][0]), 4), "intact_iou": round(float(m["iou"][1]), 4),
        "damaged_iou": round(float(m["iou"][2]), 4), "destroyed_iou": round(float(m["iou"][3]), 4),
        "precision": round(float(m["precision"].mean()), 4), "recall": round(float(m["recall"].mean()), 4),
        "f1": round(float(m["f1"].mean()), 4), "f2": round(float(m["f2"].mean()), 4),
        "destroyed_precision": round(float(m["precision"][3]), 4),
        "destroyed_recall": round(float(m["recall"][3]), 4), "destroyed_f2": round(float(m["f2"][3]), 4),
        "train_time_hrs": train_hrs, "seed": seed, "notes": notes,
    }

def describe(v):
    if v.startswith(UNET):
        return f"{UNET} pixels{' + 8-view TTA' if v.endswith('tta') else ''} (reference)"
    if v.startswith("ceiling"):
        return f"perfect grader{' with rejects' if 'rejects' in v else ''} on {v.split('_')[-1]} outlines (ceiling)"
    seed, rest = v.split("_")[1][1:], v.split("_", 2)[2]
    return (f"{TRAIN_OUTLINES} grader, seed {seed}: {rest.replace('_shares', '')} map, "
            f"{RULES['_shares'] if rest.endswith('_shares') else RULES['']}")

with open(OUT_DIR / "D4_R16c.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    for v in VARIANTS:
        seed = int(v.split("_")[1][1:]) if v.startswith("R16c_s") else None
        hrs = round(GRADER_MIN / 60 / len(SEEDS), 2) if seed else None
        writer.writerow(result_row(v, metrics(cms[v]["all"]), describe(v),
                                   "E-protocol; epoch and biases picked on held-out training images", seed, hrs))

config = {"task": "R16c", "train_outlines": TRAIN_OUTLINES, "labels": LABELS, "smoke_test": SMOKE_TEST,
          "fit_images": len(fit_pairs), "held_out_images": len(hold_pairs), "val_images": len(val_pairs),
          "sam3_dir": str(SAM3_DIR), "sam3_train_dir": str(SAM3_TRAIN_DIR) if SAM3_TRAIN_DIR else None,
          "unet": UNET, "unet_checkpoint": str(UNET_CKPT), "unet_reproduces_csv": sanity,
          "crop": CROP, "margin": MARGIN, "min_area": MIN_AREA, "max_per_image": MAX_PER_IMAGE,
          "epochs": EPOCHS, "batch": BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY, "seeds": list(SEEDS),
          "fit_buildings": dict(zip(LABELS, fit_counts.tolist())), "held_out_buildings": dict(zip(LABELS, hold_counts.tolist())),
          "real_share": dict(zip(LABELS, REAL_SHARE.round(4).tolist())), "chosen": {str(s): c for s, c in CHOSEN.items()},
          "grader_minutes": round(GRADER_MIN, 1), "validation_minutes": round(VAL_MIN, 1),
          "missing_sam3_maps": len(missing_sam), "true_buildings_macro_f1": {str(s): f for s, f in macro_f1.items()},
          "keep": verdicts, "phase_gate": gates, "gap_closed": gap_closed,
          "kept_on_all_seeds": kept_on_all_seeds, "gate_on_all_seeds": gate_on_all_seeds}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

chosen_df = pd.DataFrame([{"seed": s, **{k: (v if k != "bias" else str(v)) for k, v in c.items()}} for s, c in CHOSEN.items()])
(OUT_DIR / "README.md").write_text("\n".join([
    f"# R16c results ({TRAIN_OUTLINES}): a stronger R16 grader", "",
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}. The finding as in R16: SAM 3 `pre_grid2` "
    f"(E3) plus {UNET}'s building pieces that SAM 3 missed. Grader: Siamese ResNet-18, {CROP} px crops with a "
    f"{MARGIN} px margin, trained on {'the true pieces' if TRAIN_OUTLINES == 'truth' else 'SAM 3 outlines'} of "
    f"{len(fit_pairs):,} training images; the epoch and the label biases were picked on {len(hold_pairs):,} held-out "
    f"training images; seeds {list(SEEDS)}.", "",
    "## Pixel scores (E-protocol)", "",
    "`_shares` rows use R16b's fixed rule instead of the searched biases. `ceiling` rows give each found building "
    "its main true grade (`rejects`: or background).", "",
    as_markdown(overall), "",
    f"{UNET} reproduces its D4 CSV: **{sanity}**.", "",
    f"## Keep rule against {UNET} TTA (searched biases)", "",
    *[f"- {v}: {'KEEP' if k else 'not kept'}; Phase Gate {'pass' if gates[v] else 'fail'}"
      + (f"; {gap_closed[v]:.0%} of the gap from R16b to the ceiling" if v in gap_closed else "")
      for v, k in verdicts.items()], "",
    f"Kept on every seed: {kept_on_all_seeds}. Phase Gate on every seed: {gate_on_all_seeds}.", "",
    "## Picked on the held-out training images", "", as_markdown(chosen_df), "",
    "## The graders on the true validation buildings (object level)", "", as_markdown(true_df), "",
    f"Macro F1 per seed: {macro_f1} (R16: 0.656; BDAChat reports 0.763 on its own split).", "",
    "## Files", "",
    "- `D4_R16c.csv` (D4 format), `R16c_summary.csv` (also per source), `grader_true_buildings.csv`",
    "- `instances_val.csv`: one row per found building (each seed's log-probabilities, R11's and the truth's pixels)",
    "- `calibration.json` (the kept epoch and biases), `history.json`, `grader_seed*.pth`, `config.json`", "",
    "## Benchmark use", "",
    "SAM 3 isn't trained and the graders saw DisasterM3's training images only. With R11 (never trained on benchmark "
    "pictures) these outputs may answer benchmark questions; with R12 they may not.",
]), encoding="utf-8")
print(f"\n✓ Wrote {OUT_DIR}: D4_R16c.csv, R16c_summary.csv, grader_true_buildings.csv, instances_val.csv, "
      "calibration.json, history.json, config.json, README.md")

                 variant   miou  background_iou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall
                 R11_std 0.5811          0.9750      0.6753       0.3828         0.2911        0.7356               0.5379            0.3882
                 R11_tta 0.5875          0.9759      0.6821       0.3956         0.2965        0.7420               0.5518            0.3906
            ceiling_sam3 0.7060          0.9685      0.6243       0.6755         0.5555        0.6304               0.8401            0.6212
        ceiling_sam3+r11 0.7657          0.9752      0.7382       0.7359         0.6136        0.7426               0.7296            0.7942
ceiling_rejects_sam3+r11 0.7981          0.9816      0.7888       0.7616         0.6606        0.7907               0.8632            0.7378
           R16c_s42_sam3 0.5470          0.9685      0.5769       0.3436         0.2991        0.6304               0.5129            0.4177
       R16c_s